<a href="https://colab.research.google.com/github/Amazingsai/Pretraining-a-Small-Language-Model-from-Scratch/blob/main/CMPSC_448_Homework_3_words.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
!pip install datasets==3.6.0

In [39]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from datasets import load_dataset

# Making the results more repeatable
np.random.seed(10)
tf.random.set_seed(10)


In [40]:
from huggingface_hub import login
login()

In [41]:
from datasets import load_dataset

dataset = load_dataset("GEM/viggo",trust_remote_code=True)

print(dataset)
print("Available columns:", dataset["train"].column_names)
print("Example:", dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 5103
    })
    validation: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 714
    })
    test: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 1083
    })
    challenge_train_1_percent: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 50
    })
    challenge_train_2_percent: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 103
    })
    challenge_train_5_percent: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 256
    })
    challenge_train_10_percent: Dataset({
        features: ['gem_id', 'meaning_representation', 'target', 'references'],
        num_rows: 5

In [42]:
# Collecting the text from the target column
texts = []

for row in dataset["train"]:
  sentance = row["target"]
  if sentance != "":
    texts.append(sentance)
texts = texts[:1500]
text = "\n".join(texts)


In [43]:
# Turning words into numbers
# each word as an individual token

tokenizer = Tokenizer()



In [44]:
#Training the tokenizer using text from the target column
tokenizer.fit_on_texts([text])

In [56]:
#converting the text into a sequence of numeric IDs.
encoded_text = tokenizer.texts_to_sequences([text])[0]

In [46]:
#counts the words in the vocabulary and adds one because Keras reserves ID 0 for padding
vocab_size = len(tokenizer.word_index) + 1

In [47]:
print("Number of different characters:", vocab_size - 1)
print("First 30 character IDs:", encoded_text[:30])

Number of different characters: 1409
First 30 character IDs: [458, 459, 35, 156, 1, 3, 108, 61, 5, 8, 4, 22, 32, 17, 39, 83, 199, 8, 91, 199, 6, 336, 47, 42, 26, 9, 25, 38, 49, 33]


In [48]:
#creating training samples
#the model sees 20 words and learns to predict the next word.
sequence_length = 20
step = 3

x = []
y = []

for i in range(0, len(encoded_text) - sequence_length, step):
    x.append(encoded_text[i:i + sequence_length])
    y.append(encoded_text[i + sequence_length])

x = np.array(x)
y = np.array(y)

print("Input examples shape:", x.shape)
print("Target examples shape:", y.shape)

Input examples shape: (11126, 20)
Target examples shape: (11126,)


In [49]:
#Building th RNN model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=32
    ),
    tf.keras.layers.SimpleRNN(64),
    tf.keras.layers.Dense(
        vocab_size,
        activation="softmax"
    )
])

In [50]:
#Compiling the model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [51]:
#Training the model
history = model.fit(x,y,epochs=10,batch_size=64,validation_split=0.1)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 9s 35ms/step - accuracy: 0.0290 - loss: 5.9579 - val_accuracy: 0.0243 - val_loss: 5.7468
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0364 - loss: 5.6158 - val_accuracy: 0.0279 - val_loss: 5.8256
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.0517 - loss: 5.5049 - val_accuracy: 0.0530 - val_loss: 5.5581
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0800 - loss: 5.2481 - val_accuracy: 0.0854 - val_loss: 5.3537
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1316 - loss: 4.9530 - val_accuracy: 0.1213 - val_loss: 5.1109
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.1663 - loss: 4.6369 - val_accuracy: 0.1500 - val_loss: 4.8873
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.1992 - loss: 4.3536 - val_accuracy: 0.1680 - val_loss: 4.7056
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.2379 - loss: 4.1100 - val_accu

In [52]:
# Generating text from a prompt

#Reversing the tokenizer's dictionary character-to-number mapping
#so the program can convert numbers back into words
id_to_word = {
    number: word
    for word, number in tokenizer.word_index.items()
}


#To generate text, the model predicts a word, then sees the updated text, and predicts another word.
def generate_text(prompt, number_of_words=50):
    generated = prompt

    for _ in range(number_of_words):
      # Converting the generated text into word IDs
      input_ids = tokenizer.texts_to_sequences([generated])[0]

      # Keep only the most recent 20 words
      input_ids = input_ids[-sequence_length:]


      # Padding by adding zero IDs at the beginning if the prompt is shorter than 20 words
      if len(input_ids) < sequence_length:
            input_ids = [0] * (sequence_length - len(input_ids)) + input_ids

      #Making the prediction
      #The input IDs are converted into an array and passed through the trained model.
      #producing probabilities for the possible next words.
      predictions = model.predict(np.array([input_ids]), verbose=0)[0]

      # Adjusting the prediction probabilities
      #Using temperature sampling to control how predictable or random the next word will be when the model generates text
      temperature = 0.5
      predictions = np.log(predictions + 1e-8) / temperature
      predictions = np.exp(predictions)
      predictions = predictions / np.sum(predictions)

      # Randomly chooses the next word for the language model to generate.
      next_id = int(np.random.choice(len(predictions), p=predictions))


      #Converting the ID back into a word
      #If the predicted ID is invalid for this mapping, the loop stops.
      #But otherwise, the program converts the ID into a word and appends it to the generated text.
      if next_id == 0 or next_id not in id_to_word:
          break

      generated += " " + id_to_word[next_id]

    #This returns the original prompt plus the generated continuation.
    return generated




In [53]:
# Try a few different prompts
print("PROMPT 1")
print(generate_text("The character is the best", 50))
print()

print("PROMPT 2")
print(generate_text("I think I like", 50))
print()

print("PROMPT 3")
print(generate_text("This game", 50))

print("PROMPT 4")
print(generate_text("The game", 50))


PROMPT 1
The character is the best are you mean to the game is an average game that with a first person perspective that you remember that you can play in playstation and pc and pc and available on steam and is a third person game that you mentioned you mentioned for the game of the game

PROMPT 2
I think I like to the first person perspective or nintendo and mac it is available on steam you can played for playstation xbox and pc and pc and is available on steam and and pc it is a pc and pc and pc and is a thief's release that i think you think

PROMPT 3
This game is an action adventure game that you played and the game is an average game for playstation and third person action adventure game that is a third person perspective the game is a good driving strategy game that is not in 2014 and mac by corsa is a bird view
PROMPT 4
The game is a good strategy game with pc and pc and is a multiplayer adventure game that is a side view perspective from out in the game is a good adventure game

In [54]:
model.save("simple_game_text_rnn.keras")
print("Saved model as simple_game_text_rnn.keras")

Saved model as simple_game_text_rnn.keras
